# Fig S7: AFR and EUR studies with matched and cross-population LD

Use single-population studies from `13` `[EUR_AFR]` (`NEW_EUR_AFR_Mix_simulation`, n = 800): `prop_1.0` (all AFR) and `prop_0.0` (all EUR). Each study is fine-mapped with the matched and cross-population LD cached by notebook 13 (AFR from 1000 Genomes; EUR from UK Biobank) (`panel_LD/EUR_KG_<i>.panelLD.rds`, cached by `13`), using SuSiE-RSS and SuSiE-rRSS (`R_finite` = panel size, `R_mismatch = "eb_mix"`). Use the same $z$-scores and variants for both LD comparisons.

- `[cross_ref_finemap]`: fits `rss_{AFR,EUR}_LD` and `rrss_{AFR,EUR}_LD`; h2g 0.025 / 0.05 / 0.10 / 0.15 (800 × h2g = 20 / 40 / 80 / 120), 1–3 causal.
- `[cross_ref_summary]`: power / FDR counts per fit and SER fallback model, and the fallback flag.

Output: `/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/AFR_EUR_cross_ref_fine/`, read by `Data_Processing/Figure_4/FigS7.AFR_EUR_Cross_Reference`.

**Sketch dimension and identity LD.** Compare SER fallback across `R_finite` settings and reference matrices. Also compare the AFR and EUR panels with identity LD.
- `[cross_ref_B_sweep]`: the same studies refit by SuSiE-rRSS with AFR, EUR or identity LD and `R_finite` = panel size (AFR / EUR only), 800 or 10,000; flag, `Q_art`, `D_CS,max` and credible-set counts per fit (`n_rep` replicates per setting).
- `[cross_ref_ld_similarity]`: per region, MSE, max |ΔLD| and correlation of EUR vs AFR LD and identity vs AFR (or EUR) LD, over pairs with |r| > 0.05 in the study LD (as Fig 3B) and over all pairs.

Output: `/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/AFR_EUR_cross_ref_Bsweep/`, read by `Data_Processing/Figure_4/FigS7.AFR_EUR_B_Identity_Check`.


## Fine-mapping

In [ ]:
[cross_ref_finemap]
# Single-population studies from 13 [EUR_AFR] (prop 1.0 = all AFR, prop 0.0 = all EUR), fine-mapped with the matched
# and the cross-population 1000 Genomes panel LD, by SuSiE-RSS and SuSiE-rRSS (R_finite = panel size, eb_mix).
parameter: result_root = path("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_Mix_simulation/result")
parameter: cwd = path("output")
parameter: h2g    = "0.05"
parameter: causal = "1"
parameter: prop   = "1.0"
# AFR sample-size proportion: 1.0 (AFR study) or 0.0 (EUR study)
parameter: job_size = 30
parameter: walltime = "40h"
parameter: mem = "30G"
parameter: numThreads = 1
parameter: container = ""
input: [str(x) for x in path(f"{result_root}/h2g_{h2g}/causal_{causal}/prop_{prop}").glob("sample_*_mix_pop.rds")], group_by = 1
output: f'{cwd:a}/h2g_{h2g}/causal_{causal}/prop_{prop}/{_input[0]:bn}.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads,
      tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(Rfast)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select

    # Project utilities
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")
    source_build <- function(dir) {
      env <- new.env(parent = globalenv())
      for (f in list.files(dir, pattern = "\\.R$", full.names = TRUE)) sys.source(f, envir = env)
      env
    }
    susie_hc <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie_rss

    # Safe wrapper: takes the susie fn explicitly, catches errors, returns NULL
    safe_susie <- function(.fn, .label, ...) {
      tryCatch(
        withCallingHandlers(
          .fn(...),
          warning = function(w) invokeRestart("muffleWarning")
        ),
        error = function(e) {
          cat(sprintf("[FAIL] %s: %s\n", .label, conditionMessage(e)))
          NULL
        }
      )
    }

    ## ---- load sim output (13 [EUR_AFR]) and the cached 1000G panel LD ----
    ld_path <- ${_input:r}
    dats    <- readRDS(ld_path)
    prop    <- "${prop}"
    study   <- if (dats$n_afr > 0 && dats$n_eur == 0) "AFR" else if (dats$n_eur > 0 && dats$n_afr == 0) "EUR" else
                 stop("expected a single-population study (prop 0.0 or 1.0)")
    tb      <- if (study == "AFR") dats$afr_target_tb else dats$eur_target_tb
    n       <- dats$total_n

    panel     <- readRDS(dats$panel_ld_path)
    ref_order <- panel$variant_id                       # = columns of both panels and of the simulated genotypes
    z         <- tb$z[match(ref_order, tb$variant_id)]
    if (anyNA(z)) stop(sprintf("%d panel variants missing from the sumstats", sum(is.na(z))))
    LD <- list(AFR = panel$AFR_LD_full[ref_order, ref_order], EUR = panel$EUR_LD_full[ref_order, ref_order])
    B  <- c(AFR = panel$n_afr_panel, EUR = panel$n_eur_panel)   # panel sizes -> R_finite

    ## ---- SuSiE-RSS and SuSiE-rRSS with matched and cross-population LD ----
    result_list <- list()
    for (ld in c("AFR", "EUR")) {
      result_list[[paste0("rss_", ld, "_LD")]]  <- safe_susie(susie_hc, paste("rss", ld),  z = z, R = LD[[ld]], n = n)
      result_list[[paste0("rrss_", ld, "_LD")]] <- safe_susie(susie_hc, paste("rrss", ld), z = z, R = LD[[ld]], n = n,
                                                             R_finite = B[[ld]], R_mismatch = "eb_mix")
    }

    output <- list(
      result_list = result_list,
      study       = study,               # population of the summary statistics
      n_sample    = n,
      R_finite    = B,
      prop        = prop, h2g = "${h2g}", causal = "${causal}",
      input_path  = ld_path,
      true_vars   = dats$true_vars
    )
    saveRDS(output, ${_output:r})
    ncs <- function(f) if (is.null(f)) NA else length(f$sets$cs)
    flag <- function(f) if (is.null(f)) NA else isTRUE(f$R_finite_diagnostics$R_reliability_flag)
    cat(sprintf("[OK] %s study: CS rss(AFR=%s EUR=%s) rrss(AFR=%s EUR=%s); fallback AFR LD=%s EUR LD=%s\n", study,
                ncs(result_list$rss_AFR_LD), ncs(result_list$rss_EUR_LD), ncs(result_list$rrss_AFR_LD),
                ncs(result_list$rrss_EUR_LD), flag(result_list$rrss_AFR_LD), flag(result_list$rrss_EUR_LD)))

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="AFR_EUR_cross_ref_fine"
job_dir="${base_dir}/${job_name}"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME_c_CAUSAL_h_H2G_p_PROP
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 42:00
#BSUB -o /WORK_DIR/log/JOB_NAME_c_CAUSAL_h_H2G_p_PROP.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME_c_CAUSAL_h_H2G_p_PROP.%J.err
export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/28.FigS7.AFR_EUR_Cross_Reference.ipynb" cross_ref_finemap \
    --h2g H2G --causal CAUSAL --prop PROP \
    --cwd WORK_DIR/result \
    --mem 30G
EOF
chmod +x ${job_name}.sh
base_sh=${job_name}.sh
work_dir="${job_dir}"

# single-population studies only: prop 1.0 = AFR study, prop 0.0 = EUR study (folders written by 13 [EUR_AFR])
for causal in 1 2 3; do
  for h2g in 0.025 0.05 0.10 0.15; do
    for prop in 0.0 1.0; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}_prop_${prop}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|PROP|${prop}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"
      sb "${output_script}"
    done
  done
done

## Summary

In [ ]:
[cross_ref_summary]
# One row per (simulation, fit) and per SER fallback sub-model ("<fit>_SER"), with the fallback flag.
parameter: folder = path
parameter: cwd = path("output")
parameter: job_size = 1
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 1
parameter: container = ""
output: f'{cwd:a}/{folder:b}_{step_name}.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(tidyverse)
    `%||%` <- function(a, b) if (is.null(a)) b else a

    # true_set_num / discover_var_num as in 10 [susie_bias_summary]
    true_set_num <- function(variant, cs) {
      if (length(cs) == 0) return(0L)
      sum(vapply(cs, function(x) length(intersect(variant, x)) > 0, logical(1)))
    }
    discover_var_num <- function(variant, cs) {
      if (length(cs) == 0) return(0L)
      length(intersect(variant, unlist(cs)))
    }

    files <- list.files(${folder:r}, pattern = "sample_.*\\.rds$", recursive = TRUE, full.names = TRUE)

    tb <- map_dfr(files, function(f) {
      res     <- readRDS(f)
      variant <- res$true_vars
      row_from_fit <- function(fit, label, diag) {
        cs <- fit$sets$cs
        tibble(
          input_file         = f,
          study              = res$study,
          h2g                = as.numeric(res$h2g),
          causal             = as.integer(res$causal),
          method             = label,
          true_var_number    = discover_var_num(variant, cs),
          true_set_number    = true_set_num(variant, cs),
          total_var_number   = length(variant),
          total_set_number   = length(cs),
          total_set_length   = length(unlist(cs)),
          Q_art              = diag$Q_art              %||% NA_real_,
          D_CS_max           = if (!is.null(diag$bf_attenuation$cs_max))
                                 suppressWarnings(max(diag$bf_attenuation$cs_max, na.rm = TRUE)) else NA_real_,
          R_reliability_flag = diag$R_reliability_flag %||% NA
        )
      }
      map_dfr(names(res$result_list), function(m) {
        sus <- res$result_list[[m]]
        if (is.null(sus)) return(tibble(input_file = f, study = res$study, h2g = as.numeric(res$h2g),
                                        causal = as.integer(res$causal), method = m, fit_failed = TRUE))
        diag <- sus$R_finite_diagnostics
        rows <- row_from_fit(sus, m, diag)
        if (!is.null(diag$ser_model))
          rows <- bind_rows(rows, row_from_fit(diag$ser_model, paste0(m, "_SER"), diag$ser_model$R_finite_diagnostics))
        rows
      })
    })
    if (!"fit_failed" %in% names(tb)) tb$fit_failed <- FALSE
    tb <- mutate(tb, fit_failed = fit_failed %in% TRUE)
    saveRDS(tb, ${_output:r})

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/AFR_EUR_cross_ref_fine/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J cross_ref_summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/summary.%J.out
#BSUB -e WORK_DIR/summary/summary.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/28.FigS7.AFR_EUR_Cross_Reference.ipynb" cross_ref_summary \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary
EOF

cat base_script | sed "s|WORK_DIR|${work_dir}|g" > summary_cross_ref.sh
sb summary_cross_ref.sh

## Sketch dimension and identity LD: SER fallback

Use the studies and $z$-scores from `[cross_ref_finemap]`. Each replicate is refit by SuSiE-rRSS (`R_mismatch = "eb_mix"`) with
- LD = AFR, EUR (panels cached by `13`) × `R_finite` = panel size, 800, 10,000;
- LD = identity × `R_finite` = 800, 10,000.

Save per-fit summaries (flag, `Q_art`, `D_CS,max`, credible sets and true sets of the fit and of its SER model). `n_rep` replicates per setting (files in sorted order).

In [ ]:
[cross_ref_B_sweep]
parameter: result_root = path("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_Mix_simulation/result")
parameter: cwd = path("output")
parameter: h2g    = "0.05"
parameter: causal = "1"
parameter: prop   = "1.0"
# AFR sample-size proportion: 1.0 (AFR study) or 0.0 (EUR study)
parameter: B_fixed = "893,10000"      # fixed R_finite values, comma-separated
parameter: n_rep = 200
parameter: job_size = 20
parameter: walltime = "40h"
parameter: mem = "30G"
parameter: numThreads = 1
parameter: container = ""
input: sorted(str(x) for x in path(f"{result_root}/h2g_{h2g}/causal_{causal}/prop_{prop}").glob("sample_*_mix_pop.rds"))[:n_rep], group_by = 1
output: f'{cwd:a}/h2g_{h2g}/causal_{causal}/prop_{prop}/{_input[0]:bn}.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads,
      tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(Rfast)
    `%||%` <- function(a, b) if (is.null(a)) b else a

    # susieR GitHub build (R_finite / eb_mix), as in [cross_ref_finemap]
    source_build <- function(dir) {
      env <- new.env(parent = globalenv())
      for (f in list.files(dir, pattern = "\\.R$", full.names = TRUE)) sys.source(f, envir = env)
      env
    }
    susie_hc <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie_rss
    select <- dplyr::select

    ## ---- study z and the cached panel LD (as [cross_ref_finemap]) ----
    dats  <- readRDS(${_input:r})
    study <- if (dats$n_afr > 0 && dats$n_eur == 0) "AFR" else if (dats$n_eur > 0 && dats$n_afr == 0) "EUR" else
               stop("expected a single-population study (prop 0.0 or 1.0)")
    tb    <- if (study == "AFR") dats$afr_target_tb else dats$eur_target_tb
    n     <- dats$total_n
    panel     <- readRDS(dats$panel_ld_path)
    ref_order <- panel$variant_id
    z <- tb$z[match(ref_order, tb$variant_id)]
    if (anyNA(z)) stop(sprintf("%d panel variants missing from the sumstats", sum(is.na(z))))
    p  <- length(ref_order)
    LD <- list(AFR = panel$AFR_LD_full[ref_order, ref_order], EUR = panel$EUR_LD_full[ref_order, ref_order],
               Identity = diag(p))
    B_panel <- c(AFR = panel$n_afr_panel, EUR = panel$n_eur_panel)

    ## settings: AFR / EUR LD x (panel, B_fixed); identity x B_fixed
    B_fixed  <- as.numeric(strsplit("${B_fixed}", ",")[[1]])
    settings <- bind_rows(
      expand_grid(ld = c("AFR", "EUR"), B_setting = c("panel", as.character(B_fixed))),
      expand_grid(ld = "Identity",      B_setting = as.character(B_fixed))) %>%
      mutate(B = if_else(B_setting == "panel", as.numeric(unname(B_panel[ld])), suppressWarnings(as.numeric(B_setting))))

    true_set_num <- function(variant, cs) if (!length(cs)) 0L else sum(vapply(cs, function(x) length(intersect(variant, x)) > 0, logical(1)))
    fit_row <- function(fit) {
      diag <- fit$R_finite_diagnostics
      ser  <- diag$ser_model
      tibble(R_reliability_flag = isTRUE(diag$R_reliability_flag),
             Q_art     = diag$Q_art %||% NA_real_,
             D_CS_max  = if (!is.null(diag$bf_attenuation$cs_max)) suppressWarnings(max(diag$bf_attenuation$cs_max, na.rm = TRUE)) else NA_real_,
             n_cs      = length(fit$sets$cs),
             n_true_cs = true_set_num(dats$true_vars, fit$sets$cs),
             n_cs_ser      = if (is.null(ser)) NA_integer_ else length(ser$sets$cs),
             n_true_cs_ser = if (is.null(ser)) NA_integer_ else true_set_num(dats$true_vars, ser$sets$cs))
    }

    rows <- pmap_dfr(settings, function(ld, B_setting, B) {
      fit <- tryCatch(suppressWarnings(susie_hc(z = z, R = LD[[ld]], n = n, R_finite = B, R_mismatch = "eb_mix")),
                      error = function(e) { cat(sprintf("[FAIL] %s LD, B = %s: %s\n", ld, B, conditionMessage(e))); NULL })
      base <- tibble(input_file = ${_input:r}, study = study, h2g = as.numeric("${h2g}"), causal = as.integer("${causal}"),
                     n = n, p = p, ld = ld, B_setting = B_setting, B = B, n_causal = length(dats$true_vars))
      if (is.null(fit)) return(mutate(base, fit_failed = TRUE))
      bind_cols(base, fit_row(fit)) %>% mutate(fit_failed = FALSE)
    })
    saveRDS(rows, ${_output:r})
    print(as.data.frame(select(rows, study, ld, B_setting, B, any_of(c("R_reliability_flag", "Q_art", "D_CS_max", "n_cs")))))

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="AFR_EUR_cross_ref_Bsweep"
job_dir="${base_dir}/${job_name}"

mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME_c_CAUSAL_h_H2G_p_PROP
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 42:00
#BSUB -o /WORK_DIR/log/JOB_NAME_c_CAUSAL_h_H2G_p_PROP.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME_c_CAUSAL_h_H2G_p_PROP.%J.err
export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/28.FigS7.AFR_EUR_Cross_Reference.ipynb" cross_ref_B_sweep \
    --h2g H2G --causal CAUSAL --prop PROP --n_rep 200 \
    --cwd WORK_DIR/result \
    --mem 30G
EOF
chmod +x ${job_name}.sh
base_sh=${job_name}.sh
work_dir="${job_dir}"

for causal in 1 2 3; do
  for h2g in 0.025 0.05 0.10 0.15; do
    for prop in 0.0 1.0; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}_prop_${prop}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|PROP|${prop}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"
      sb "${output_script}"
    done
  done
done

## LD similarity of AFR, EUR and identity LD

Per region (`panel_LD/EUR_KG_<i>.panelLD.rds` cached by `13`), with the study's own panel LD as the reference point (AFR study: AFR LD; EUR study: EUR LD) and each other LD compared with it: the other population's LD and the identity matrix. Metrics as Fig 3B (`09` `[LD_mismatch]`): MSE, max |ΔLD| and correlation of the upper-triangle entries, over pairs with |r| > 0.05 in the study LD (`pairs = "study_r_gt_0.05"`) and over all pairs (`pairs = "all"`). The correlation with identity is undefined (its off-diagonal entries are all 0) and is left `NA`; MSE against identity over all pairs is the panel's mean off-diagonal r².

In [ ]:
[cross_ref_ld_similarity]
parameter: panel_dir = path("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_Mix_simulation/panel_LD")
parameter: cwd = path("output")
parameter: ld_thresh = 0.05
parameter: job_size = 1
parameter: walltime = "10h"
parameter: mem = "30G"
parameter: numThreads = 1
parameter: container = ""
input: sorted(str(x) for x in panel_dir.glob("EUR_KG_*.panelLD.rds")), group_by = 50
output: f'{cwd:a}/ld_similarity_chunk_{_index}.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads,
      tags = f'{step_name}_{_index}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(tidyverse)

    ## metrics of a reference LD against the study LD on the upper triangle (as 09 [LD_mismatch])
    ld_metrics <- function(R_ref, R_study, filter) {
      ut <- upper.tri(R_study); a <- R_ref[ut]; b <- R_study[ut]
      ok <- is.finite(a) & is.finite(b)
      if (filter) ok <- ok & abs(b) > ${ld_thresh}
      d  <- a[ok] - b[ok]
      tibble(mse = mean(d^2), max_abs_diff = max(abs(d)),
             concord = if (sd(a[ok]) > 0) suppressWarnings(cor(a[ok], b[ok])) else NA_real_,
             n_pairs = sum(ok))
    }

    files <- c(${_input:r,})
    rows <- map_dfr(files, function(f) {
      panel <- readRDS(f)
      v  <- panel$variant_id
      LD <- list(AFR = panel$AFR_LD_full[v, v], EUR = panel$EUR_LD_full[v, v])
      LD$Identity <- diag(length(v))
      region <- str_remove(basename(f), "\\.panelLD\\.rds$")
      map_dfr(c("AFR", "EUR"), function(study) {
        map_dfr(setdiff(names(LD), study), function(ref) {
          map_dfr(c(study_r_gt_0.05 = TRUE, all = FALSE), function(flt)
            ld_metrics(LD[[ref]], LD[[study]], flt), .id = "pairs") %>%
            mutate(region = region, study = study, reference = ref, p = length(v), .before = 1)
        })
      })
    })
    saveRDS(rows, ${_output:r})

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/AFR_EUR_cross_ref_Bsweep"
mkdir -p ${work_dir}/{code,ld_similarity,log}
cd ${work_dir}/code

cat << 'EOF' > ld_similarity.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J AFR_EUR_ld_similarity
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 24:00
#BSUB -o WORK_DIR/log/ld_similarity.%J.out
#BSUB -e WORK_DIR/log/ld_similarity.%J.err
export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/28.FigS7.AFR_EUR_Cross_Reference.ipynb" cross_ref_ld_similarity \
    --cwd WORK_DIR/ld_similarity --mem 30G -J 4
EOF
sed "s|WORK_DIR|${work_dir}|g" ld_similarity.sh > ld_similarity.sh.submit
sb ld_similarity.sh.submit